# Cell Segmentation Pipeline: CLAHE → Segmentation → Volume Filtering

Three-step pipeline:
1. **3D CLAHE** — contrast-enhance raw TIFF stacks
2. **Bulk segmentation** — run custom Cellpose model on CLAHE stacks
3. **Post-processing** — measure cell volumes across the whole dataset, plot the distribution, and flag small/large outliers using a 3-class (multi-) Otsu threshold


## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tiff
from tqdm import tqdm

from skimage.exposure import equalize_adapthist
from skimage import measure
from skimage.filters import threshold_multiotsu
from cellpose import models


## Config

All paths and parameters live here — edit once, run the whole pipeline below.

In [ ]:
# --- directories (EDIT pw)---
pw = '/Users/claudia/Desktop/data_from_others/vesta_ctBpalm_crbGFP_cpv3/' # EDIT this pathway for your project directory 
raw_dir   = pw + "raw_tiffs"                    # input: raw TIFF stacks
clahe_dir = pw +  "clahe_tiffs"                 # output of step 1 / input to step 2
mask_dir  = pw + "mask_tiffs"                   # output of step 2 / input to step 3
results_dir = pw + "results"                    # csvs + plots from step 3

for d in (clahe_dir, mask_dir, results_dir):
    os.makedirs(d, exist_ok=True)

# --- CLAHE parameters ---
clip_limit = 0.01
kernel_size = None      # or e.g. (8, 64, 64)
output_dtype = np.uint16

# --- Cellpose model + inference parameters (EDIT model_path) ---
model_path = "/Users/claudia/Documents/github_repo/cell_segmentation_models/models/cell_seg_v3_masksonly_epoch_0190" # EDIT with location of model_path
stitch_threshold = 0.4
cellprob_threshold = 0.0
flow_threshold = 0.2

# --- voxel spacing (Z, Y, X) in microns — UPDATE to your imaging metadata ---
voxel_spacing = (1.5, 0.431583728751555, 0.431583728751555)


## Step 1: 3D CLAHE

In [ ]:
tiff_files = sorted([
    f for f in os.listdir(raw_dir)
    if f.lower().endswith((".tif", ".tiff"))
])
print(f"Found {len(tiff_files)} TIFF files.")


In [ ]:
for filename in tqdm(tiff_files, desc="CLAHE"):
    input_path = os.path.join(raw_dir, filename)
    output_path = os.path.join(clahe_dir, filename)

    if os.path.exists(output_path):
        continue  # skip already-processed files

    volume = tiff.imread(input_path).astype(np.float32)
    volume -= volume.min()
    if volume.max() > 0:
        volume /= volume.max()

    volume = equalize_adapthist(volume, clip_limit=clip_limit, kernel_size=kernel_size)

    if output_dtype == np.uint8:
        volume = (volume * 255).astype(np.uint8)
    elif output_dtype == np.uint16:
        volume = (volume * 65535).astype(np.uint16)

    tiff.imwrite(output_path, volume)

print("✓ CLAHE done.")


## Step 2: Bulk segmentation (custom Cellpose model)

In [ ]:
clahe_files = sorted([
    f for f in os.listdir(clahe_dir)
    if f.lower().endswith((".tif", ".tiff"))
])
print(f"Found {len(clahe_files)} CLAHE TIFF files.")

model = models.CellposeModel(gpu=True, pretrained_model=model_path)


In [ ]:
for filename in tqdm(clahe_files, desc="Segmenting"):
    input_path = os.path.join(clahe_dir, filename)
    output_path = os.path.join(mask_dir, filename.replace(".tif", "_masks.tif"))

    if os.path.exists(output_path):
        continue  # skip already-segmented files

    volume = tiff.imread(input_path)

    masks, flows, styles = model.eval(
        volume,
        diameter=None,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        do_3D=False,                       # 2D per-slice, matching training
        stitch_threshold=stitch_threshold, # links masks across z
        channels=[0, 0],
        z_axis=0,
    )

    tiff.imwrite(output_path, masks.astype(np.uint16), compression="zlib")

print("✓ Segmentation done.")


## Step 3: Post-processing — volume distribution + small/large filtering

Measures every cell's volume (in µm³, using `voxel_spacing`) across **all** mask files, plots the distribution for the whole dataset, then uses a 3-class Otsu threshold (`threshold_multiotsu`) to split volumes into **small / normal / large** — flagging small objects (likely debris/fragments) and large objects (likely merged cells) in one step.

In [ ]:
mask_files = sorted([
    f for f in os.listdir(mask_dir)
    if f.lower().endswith((".tif", ".tiff"))
])
print(f"Found {len(mask_files)} mask files.")

voxel_volume = voxel_spacing[0] * voxel_spacing[1] * voxel_spacing[2]  # µm³ per voxel

records = []
for filename in tqdm(mask_files, desc="Measuring volumes"):
    masks = tiff.imread(os.path.join(mask_dir, filename))
    props = measure.regionprops(masks, spacing=voxel_spacing)
    for prop in props:
        volume_um3 = prop.area  # already in µm³ because spacing was passed
        records.append({
            "filename": filename,
            "cell_id": prop.label,
            "volume_um3": volume_um3,
            "voxel_count": int(round(volume_um3 / voxel_volume)),  # raw voxel count
        })

df = pd.DataFrame(records)
print(f"Total cells across dataset: {len(df)}")
print(df[["volume_um3", "voxel_count"]].describe())


In [ ]:
# --- volume (µm³) and voxel count distributions across the whole dataset ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df["volume_um3"], bins=60, color="steelblue", edgecolor="black")
axes[0].set_xlabel("Volume (µm³)")
axes[0].set_ylabel("Cell count")
axes[0].set_title(f"Volume distribution (n={len(df)})")

axes[1].hist(df["voxel_count"], bins=60, color="seagreen", edgecolor="black")
axes[1].set_xlabel("Voxel count")
axes[1].set_ylabel("Cell count")
axes[1].set_title(f"Voxel count distribution (n={len(df)})")

plt.tight_layout()
plt.show()


In [ ]:
# --- 3-class (multi-) Otsu: splits volumes into small / normal / large ---
# (computed on volume_um3; voxel_count is just a linear rescaling of the same
#  distribution, so the same split applies — shown below in both units)
volumes = df["volume_um3"].values
small_thresh, large_thresh = threshold_multiotsu(volumes, classes=3)
small_thresh_vox = small_thresh / voxel_volume
large_thresh_vox = large_thresh / voxel_volume

print(f"Small/normal boundary: {small_thresh:.2f} µm³  ({small_thresh_vox:.0f} voxels)")
print(f"Normal/large boundary: {large_thresh:.2f} µm³  ({large_thresh_vox:.0f} voxels)")

df["size_class"] = pd.cut(
    df["volume_um3"],
    bins=[-np.inf, small_thresh, large_thresh, np.inf],
    labels=["small", "normal", "large"],
)
print(df["size_class"].value_counts())


In [ ]:
# --- plot with both thresholds marked ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["volume_um3"], bins=60, color="steelblue", edgecolor="black")
ax.axvline(small_thresh, color="orange", ls="--", lw=2, label=f"small cutoff = {small_thresh:.1f}")
ax.axvline(large_thresh, color="red", ls="--", lw=2, label=f"large cutoff = {large_thresh:.1f}")
ax.set_xlabel("Volume (µm³)")
ax.set_ylabel("Cell count")
ax.set_title("Cell volume distribution with multi-Otsu small/large thresholds")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- save results ---
df.to_csv(os.path.join(results_dir, "all_cell_volumes.csv"), index=False)

small_flagged = df[df["size_class"] == "small"].sort_values("volume_um3")
large_flagged = df[df["size_class"] == "large"].sort_values("volume_um3", ascending=False)

small_flagged.to_csv(os.path.join(results_dir, "flagged_small_cells.csv"), index=False)
large_flagged.to_csv(os.path.join(results_dir, "flagged_large_cells.csv"), index=False)

print(f"Saved all_cell_volumes.csv ({len(df)} cells)")
print(f"Saved flagged_small_cells.csv ({len(small_flagged)} cells, likely debris/fragments)")
print(f"Saved flagged_large_cells.csv ({len(large_flagged)} cells, likely merged cells)")


## Step 4: (optional) view ouputs with napari
Note: this cell only works if you have ONE file you're segmenting

In [ ]:
import napari

# find image file names
clahe_file = [
    f for f in os.listdir(clahe_dir)
    if f.lower().endswith((".tif", ".tiff"))
]

if len(clahe_file) != 1:
    raise ValueError(
        f"Expected exactly one TIFF in {clahe_dir}, found {len(clahe_file)}"
    )

clahe_path = os.path.join(clahe_dir, clahe_file[0])
print(clahe_path)

# load mask segmentation file
mask_file = [
    f for f in os.listdir(mask_dir)
    if f.lower().endswith((".tif", ".tiff"))
]

if len(mask_file) != 1:
    raise ValueError(
        f"Expected exactly one TIFF in {mask_dir}, found {len(mask_file)}"
    )

mask_path = os.path.join(mask_dir, mask_file[0])
print(mask_path)

# Load images
clahe_image = tiff.imread(clahe_path)
masks = tiff.imread(mask_path)

# Validate that they align
if clahe_image.shape != masks.shape:
    raise ValueError(
        "CLAHE image and masks have different shapes:\n"
        f"CLAHE:  {clahe_image.shape}\n"
        f"masks: {masks.shape}"
    )

# Labels should contain integer object IDs
if not np.issubdtype(masks.dtype, np.integer):
    masks = masks.astype(np.uint32)

# Open napari
viewer = napari.Viewer(title="Cellpose segmentation results")

viewer.add_image(
    clahe_image,
    name="CLAHE",
    scale=voxel_spacing,
    units="um",
    colormap="gray",
)

viewer.add_labels(
    masks,
    name="masks",
    scale=voxel_spacing,
    units="um",
    opacity=0.5,
)

viewer.scale_bar.visible = True
viewer.axes.visible = True
viewer.dims.axis_labels = ("z", "y", "x")